In [1]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    PROJECT_ROOT = Path("/content/emotion-dynamics-nlp")

    if not PROJECT_ROOT.exists():
        !git clone https://github.com/duckydodo/emotion-dynamics-nlp.git /content/emotion-dynamics-nlp

else:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)

Running in Colab: False
Project root: /home/chitta/Projects/emotion-dynamics-nlp


In [2]:
if IN_COLAB:
    %cd /content/emotion-dynamics-nlp

    # This project is NLP-only; these optional packages can conflict
    # with Colab's PyTorch CUDA build.
    !pip uninstall -y -q torchvision torchaudio

    !pip install -q transformers==5.15.0 accelerate==1.14.0
    !pip install -q -e .

%cd {PROJECT_ROOT}

print("Environment ready")

/home/chitta/Projects/emotion-dynamics-nlp
Environment ready


In [3]:
import json
import random

import numpy as np
import pandas as pd
import torch

from src.config import RAW_DATA_DIR, OUTPUT_DIR
from src.data.loader import load_meld
from src.data.context import build_context_examples

from src.models.transformer import (
    TransformerEmotionClassifier,
    LABELS,
)
from src.models.trainer import TransformerTrainer

print("Imports OK")

Imports OK


In [4]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GB",
    )
else:
    print("Using CPU")

PyTorch: 2.13.0+cu130
CUDA available: False
Using CPU


In [5]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [6]:
EXPERIMENT_NAME = "transformer_context_1"

MODEL_NAME = "distilbert-base-uncased"

LEARNING_RATE = 2e-5
BATCH_SIZE = 4
EPOCHS = 3
MAX_LENGTH = 128

CONTEXT_TURNS = 1
INCLUDE_SPEAKER = False

print("Experiment:", EXPERIMENT_NAME)
print("Context turns:", CONTEXT_TURNS)
print("Speaker information:", INCLUDE_SPEAKER)
print("Model:", MODEL_NAME)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Max length:", MAX_LENGTH)

Experiment: transformer_context_1
Context turns: 1
Speaker information: False
Model: distilbert-base-uncased
Learning rate: 2e-05
Batch size: 4
Epochs: 3
Max length: 128


In [7]:
if IN_COLAB:
    MELD_DATA_DIR = (
        Path("/content/drive/MyDrive/")
        / "emotion-dynamics-nlp"
        / "data"
        / "raw"
        / "meld"
    )
else:
    MELD_DATA_DIR = RAW_DATA_DIR / "meld"

print("MELD data:", MELD_DATA_DIR)

MELD data: /home/chitta/Projects/emotion-dynamics-nlp/data/raw/meld


In [8]:
dataset = load_meld(MELD_DATA_DIR)

train_df = dataset.train
dev_df = dataset.dev
test_df = dataset.test

print("Original train:", len(train_df))
print("Original dev:", len(dev_df))
print("Original test:", len(test_df))

train_context = build_context_examples(
    train_df,
    context_turns=CONTEXT_TURNS,
    include_speaker=INCLUDE_SPEAKER,
)

dev_context = build_context_examples(
    dev_df,
    context_turns=CONTEXT_TURNS,
    include_speaker=INCLUDE_SPEAKER,
)

test_context = build_context_examples(
    test_df,
    context_turns=CONTEXT_TURNS,
    include_speaker=INCLUDE_SPEAKER,
)

print()
print("Context-1 train:", len(train_context))
print("Context-1 dev:", len(dev_context))
print("Context-1 test:", len(test_context))

Original train: 9989
Original dev: 1109
Original test: 2610

Context-1 train: 8882
Context-1 dev: 993
Context-1 test: 2328


In [9]:
print("Labels:")
print(sorted(train_context["emotion"].unique()))

print("\nExpected labels:")
print(LABELS)

print("\nContext-1 train distribution:")
print(train_context["emotion"].value_counts())

Labels:
['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

Expected labels:
['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

Context-1 train distribution:
emotion
neutral     4168
joy         1523
surprise    1080
anger       1017
sadness      610
disgust      244
fear         240
Name: count, dtype: int64


In [10]:
print("===== CONTEXT SANITY CHECK =====")

for _, row in train_context.head(3).iterrows():
    print("-" * 60)
    print("Dialogue:", row["dialogue_id"])
    print("Target:", row["utterance_id"])
    print(row["context"])
    print("Target emotion:", row["emotion"])

===== CONTEXT SANITY CHECK =====
------------------------------------------------------------
Dialogue: 0
Target: 1
also I was the point person on my company’s transition from the KL-5 to GR-6 system.
You must’ve had your hands full.
Target emotion: neutral
------------------------------------------------------------
Dialogue: 0
Target: 2
You must’ve had your hands full.
That I did. That I did.
Target emotion: neutral
------------------------------------------------------------
Dialogue: 0
Target: 3
That I did. That I did.
So let’s talk a little bit about your duties.
Target emotion: neutral


In [11]:
classifier = TransformerEmotionClassifier(
    model_name=MODEL_NAME,
)

print("Model:", MODEL_NAME)
print("Device:", classifier.device)
print("Labels:", LABELS)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert-base-uncased
Device: cpu
Labels: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']


In [12]:
TINY_TRAIN_SIZE = 32
TINY_DEV_SIZE = 16

tiny_train = train_context.head(TINY_TRAIN_SIZE)
tiny_dev = dev_context.head(TINY_DEV_SIZE)

tiny_trainer = TransformerTrainer(
    classifier=classifier,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=1,
    max_length=64,
    seed=SEED,
)

tiny_history = tiny_trainer.fit(
    train_texts=tiny_train["context"],
    train_labels=tiny_train["emotion"],
    val_texts=tiny_dev["context"],
    val_labels=tiny_dev["emotion"],
)

print(tiny_history)

Epoch 1/1 | Loss: 1.9090 | Val Accuracy: 0.5000 | Val Macro F1: 0.1333
[{'epoch': 1, 'train_loss': 1.909027323126793, 'val_accuracy': 0.5, 'val_macro_f1': 0.13333333333333333, 'val_weighted_f1': 0.3333333333333333}]


In [13]:
from pathlib import Path

if IN_COLAB:
    CHECKPOINT_DIR = Path(
        "/content/drive/MyDrive/"
        "emotion-dynamics-nlp/"
        "checkpoints/"
        "transformer_context_1"
    )
else:
    CHECKPOINT_DIR = (
        OUTPUT_DIR
        / "models"
        / "transformer_context_1"
    )

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Checkpoint directory:", CHECKPOINT_DIR)

Checkpoint directory: /home/chitta/Projects/emotion-dynamics-nlp/outputs/models/transformer_context_1


In [12]:
FULL_TRAINING = True

TRAIN_TEXTS = train_context["context"]
TRAIN_LABELS = train_context["emotion"]

VAL_TEXTS = dev_context["context"]
VAL_LABELS = dev_context["emotion"]

print("Training examples:", len(TRAIN_TEXTS))
print("Validation examples:", len(VAL_TEXTS))
print("Full training:", FULL_TRAINING)

Training examples: 9989
Validation examples: 1109
Full training: True


In [ ]:
trainer = TransformerTrainer(
    classifier=TransformerEmotionClassifier(
        model_name=MODEL_NAME,
    ),
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    max_length=MAX_LENGTH,
    seed=SEED,
)

print("Full trainer ready")

In [ ]:
history = trainer.fit(
    train_texts=TRAIN_TEXTS,
    train_labels=TRAIN_LABELS,
    val_texts=VAL_TEXTS,
    val_labels=VAL_LABELS,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=True,
)

print(history)

In [ ]:
# Load the best model selected using validation Macro F1

best_checkpoint = CHECKPOINT_DIR / "best"

best_classifier = TransformerEmotionClassifier(
    model_name=MODEL_NAME,
)

best_classifier.load(best_checkpoint)

test_trainer = TransformerTrainer(
    classifier=best_classifier,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=1,
    max_length=MAX_LENGTH,
    seed=SEED,
)

test_results, test_probabilities = test_trainer.validate(
    test_context["context"],
    test_context["emotion"],
)

print("===== TEST RESULTS =====")
print(f"Accuracy    : {test_results.accuracy:.4f}")
print(f"Macro F1    : {test_results.macro_f1:.4f}")
print(f"Weighted F1 : {test_results.weighted_f1:.4f}")

In [ ]:
results = {
    "experiment": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "context_turns": CONTEXT_TURNS,
    "include_speaker": INCLUDE_SPEAKER,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "train_examples": len(train_context),
    "dev_examples": len(dev_context),
    "test_examples": len(test_context),
    "test_accuracy": test_results.accuracy,
    "test_macro_f1": test_results.macro_f1,
    "test_weighted_f1": test_results.weighted_f1,
}

if IN_COLAB:
    results_dir = (
        Path("/content/drive/MyDrive/")
        / "emotion-dynamics-nlp"
        / "results"
        / EXPERIMENT_NAME
    )
else:
    results_dir = (
        OUTPUT_DIR
        / "experiments"
        / EXPERIMENT_NAME
    )

results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

results_path = results_dir / "metrics.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved results to:", results_path)

temperary cell

In [ ]:
from src.models.transformer import TransformerEmotionClassifier
from src.models.trainer import TransformerTrainer

BASELINE_CHECKPOINT = (
    Path("/content/drive/MyDrive/")
    / "emotion-dynamics-nlp"
    / "checkpoints"
    / "transformer_baseline"
    / "best"
)

baseline_classifier = TransformerEmotionClassifier(
    model_name=MODEL_NAME,
)

baseline_classifier.load(BASELINE_CHECKPOINT)

baseline_trainer = TransformerTrainer(
    classifier=baseline_classifier,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=1,
    max_length=MAX_LENGTH,
    seed=SEED,
)

baseline_same_targets, baseline_probabilities = (
    baseline_trainer.validate(
        test_context["text"],
        test_context["emotion"],
    )
)

print("===== SAME-TARGET BASELINE =====")
print(
    f"Accuracy    : "
    f"{baseline_same_targets.accuracy:.4f}"
)
print(
    f"Macro F1    : "
    f"{baseline_same_targets.macro_f1:.4f}"
)
print(
    f"Weighted F1 : "
    f"{baseline_same_targets.weighted_f1:.4f}"
)